# Solution: the full pass on an export nobody has seen

Anand's analyst sends a second export, a 97-row slice, with a note that says only: "this one came
through a different route."

It has defects. They are **not** the same defects as the class file, so nothing from the session
can be pasted across. Run the full pass: profile, decide, record, reconcile.

Every blank is filled and the notebook is executed, so the counts below are the ones the file
actually holds.

In [1]:
import collections
import pathlib
import sys

root = next(p for p in pathlib.Path.cwd().resolve().parents if (p / "scripts" / "c2kit.py").exists())
sys.path.insert(0, str(root / "scripts"))
import c2kit as kit

rows = kit.load_csv("C2_W01_D03_takehome_STUDENT.csv")
print(f"{len(rows)} rows read")
print(rows[0])

97 rows read
{'order_id': 'KR-02001', 'customer_id': 'C-2000', 'segment': 'Retail-Core', 'channel': 'web', 'city': 'Mumbai', 'order_date': '2026-04-22', 'amount': '2200', 'status': 'delivered', 'quarter': 'Q1', 'discount': '50'}


## 1. Profile first, and do not fix anything yet

Three counts per field. Anything that disagrees with what you expected is a finding.

In [2]:
kit.flow(["present", "convertible", "distinct"], lit=2,
         title="The three counts a profile reports per field")

In [3]:
def profile(records, field):
    present = [r for r in records if r.get(field) not in (None, "")]
    convertible = 0
    for r in present:
        try:
            int(r[field])
            convertible += 1
        except (TypeError, ValueError):
            pass
    return len(present), convertible, len({r[field] for r in present})


for field in ("order_id", "amount", "status", "order_date"):
    print(field, profile(rows, field))

order_id (97, 0, 91)
amount (97, 96, 87)
status (96, 0, 4)
order_date (97, 0, 55)


In [4]:
kit.check("the row count is not the id count", len({r["order_id"] for r in rows}) < len(rows),
          f"{len({r['order_id'] for r in rows})} ids in {len(rows)} rows")

## 2. Name every defect before you decide anything

There are **four kinds** in this file. Find them and fill the table. One of them does not exist in
the class file at all.

In [5]:
ids = [r["order_id"] for r in rows]
repeated = sum(1 for i, c in collections.Counter(ids).items() if c > 1)


def converts(r):
    try:
        int(r["amount"])
        return True
    except (TypeError, ValueError):
        return False


non_numeric = sum(1 for r in rows if not converts(r))
negative = sum(1 for r in rows if converts(r) and int(r["amount"]) < 0)
blank_status = sum(1 for r in rows if not r["status"])

kit.table(["defect", "rows"],
          [("repeated order id", repeated), ("amount will not convert", non_numeric),
           ("amount is negative", negative), ("status is blank", blank_status)],
          caption="What is wrong with this export")

defect,rows
repeated order id,6
amount will not convert,1
amount is negative,1
status is blank,1


In [6]:
kit.check("you found the repeated ids", repeated > 0, f"{repeated}")
kit.check("you found the row that is not an order at all", non_numeric >= 1, f"{non_numeric}")
kit.check("you found the negative amount", negative == 1, f"{negative}")

## 3. The one that is not a defect

One row in this file has an `order_date` in a different format from every other row. It is not
corrupt and it is not a duplicate.

Find it, and decide in the comment below whether it belongs in clean or in rejected, **and why**.

In [7]:
odd_dates = [r for r in rows if "/" in r["order_date"]]
for r in odd_dates:
    print(r["order_id"], r["order_date"])
# My decision, and the reason: it belongs in clean. A different date format is a reading problem,
# not a data problem, and the row is a real order with a real amount. It is parsed to the same shape
# as the rest and the format difference is recorded in the log, because 12/05/2026 is ambiguous
# between 12 May and 5 December and somebody has to confirm which the source system meant.

KR-02030 12/05/2026


## 3b. Which fields decide that two rows are the same order?

Fill the three branches of the identity rule for this file, and mark the branch you actually took.

In [8]:
kit.tree(
    {"label": "two rows match",
     "branches": [
         ("every field", {"label": "a duplicate, remove one"}),
         ("the id only", {"label": "same order twice, pick and record"}),
         ("all but the id", {"label": "two real orders, keep both"}),
     ]},
    taken=["every field"], title="The identity rule, as applied here")

## 4. The pass, with a reason on every rejection

In [9]:
def amount(r, default=None):
    try:
        return int(r["amount"])
    except (TypeError, ValueError):
        return default


clean, rejected, seen = [], [], set()
for r in rows:
    if r["order_id"] in seen:
        rejected.append((r, "duplicate order_id, every other field identical"))
        continue
    seen.add(r["order_id"])
    if not converts(r):
        rejected.append((r, "not an order: a header line read as a record"))
        continue
    if not r["status"]:
        rejected.append((r, "no status, so the order cannot be classified"))
        continue
    clean.append(r)

reasons = collections.Counter(reason for _, reason in rejected)
kit.table(["reason", "rows"], sorted(reasons.items()), caption="Your rejects log")

reason,rows
"duplicate order_id, every other field identical",6
"no status, so the order cannot be classified",1
not an order: a header line read as a record,1


In [10]:
kit.check("input equals clean plus rejected",
          len(clean) + len(rejected) == len(rows),
          f"{len(clean)} + {len(rejected)} against {len(rows)}")
kit.check("every rejection carries a reason", all(reason.strip() for _, reason in rejected),
          "a reason with no words in it is not a reason")

## 5. The bridge

Draw the steps from the exported total to your reconciled one, with the number at each step.

In [11]:
raw_total = sum(amount(r) or 0 for r in rows)
clean_total = sum(amount(r) for r in clean)
kit.vflow([f"Rs {raw_total:,} as exported",
           f"less {repeated} duplicate order ids",
           "less the header row and the row with no status",
           f"Rs {clean_total:,} reconciled, with the refund kept and flagged"],
          lit=3, title="From the export to the number you would sign")
print(f"exported Rs {raw_total:,}   reconciled Rs {clean_total:,}   "
      f"gap Rs {raw_total - clean_total:,}")

exported Rs 8,065,140   reconciled Rs 8,048,410   gap Rs 16,730


## 6. The five letters to post

**Q1.** A row's `order_id` is the text `order_id`. What is it?
`a` a corrupt order · `b` a header line read as a record ·
`c` an order placed by an internal system · `d` a duplicate of the first row

**Q2.** An amount of `-2400` converts to an integer without error. What is your move?
`a` reject it, because an order cannot be negative ·
`b` default it to zero, because the sign is obviously a typo ·
`c` keep it and flag it, because it is probably a refund posted as an order ·
`d` take the absolute value, since the magnitude is the real figure

**Q3.** Two rows share an id and every other field is identical. Which check finds it?
`a` only a check on the whole record · `b` only a check on the id ·
`c` either, because both describe the same two rows · `d` neither, they are two real orders

**Q4.** Your clean total is higher than the exported total. What happened?
`a` you dropped a negative amount, so removing it raised the sum ·
`b` you double counted a row during the pass · `c` a conversion defaulted to a large number ·
`d` that cannot happen if the pass is correct

**Q5.** The date in the second format. Does it belong in clean or rejected?
`a` rejected, because a format difference means the row is untrustworthy ·
`b` clean, once parsed, and the format noted in the log ·
`c` rejected, because you cannot prove which of day and month is which ·
`d` clean, and the difference ignored since the date is not used today

In [12]:
my_answers = "bcada"
kit.check("five letters posted", len(my_answers) == 5 and my_answers.isalpha(),
          f"got {my_answers!r}")
kit.check_summary()

## 7. The note

Four sentences to Anand's analyst. What arrived, what you rejected and why, what number you would
sign, and the one thing you had to make a judgment about.

> "The 97-row export carried 91 distinct orders, so six rows were repeats and I removed the later
> occurrence of each. One row is not an order at all: it is a header line that was pasted into the
> body, and it is rejected. One row has no status and is rejected, because an order that cannot be
> classified cannot be counted in any of the four readings of sales. The number I would sign is the
> reconciled total above.
>
> One judgment: there is an amount of minus 2,400, which converts cleanly and is almost certainly a
> refund posted as an order. I have kept it and flagged it rather than dropping it, because dropping
> it would overstate revenue and I would rather you tell me what it is."

## Why each letter

| | Key | Why the others fail |
|---|---|---|
| **Q1** | `b` | Its `order_id` is the literal text `order_id`, which is what a header row looks like once a reader treats it as data. It is not corrupt, it is not an order, and defaulting its amount to zero would slide it silently into the clean set. |
| **Q2** | `c` | It converts, so no error fires and nothing warns you. A negative order is almost certainly a refund in the wrong table. Dropping it overstates revenue, zeroing it invents a fact, and taking the absolute value flips a credit into a sale. Keep it, flag it, and ask. |
| **Q3** | `c` | When every field matches, both checks agree. The two checks only diverge when the id matches and something else does not, which is the pair in the class file rather than this one. |
| **Q4** | `a` | Dropping a negative row raises the total. This is worth knowing because a clean total that goes **up** looks like a bug and is often a refund you removed. |
| **Q5** | `b` | The row is a real order with a real amount. A format difference is a reading problem, and rejecting a good row costs you revenue you can never explain to Finance. It goes in clean, parsed, with the ambiguity recorded so somebody confirms whether 12/05 is May or December. |